In [1]:
# Parameters
DB_PATH                   = "../../../DB/oedb_baseline_v3.db"
BENCHMARK_PATH            = "../../../data/input_data/benchmark_trainingset.xlsx"
BENCHMARK_SHEET           = "merged_answers"
MATCHED_ANSWERS_CSV       = "matched_answers.csv"
MATCHED_PARTICIPANTS_CSV  = "matched_participants.csv"
MATCHED_QUESTIONS_CSV     = "matched_questions.csv"
NOTEGROUP_ID_MIN          = 1
NOTEGROUP_ID_MAX          = 23

EVAL_FIELDS = ["participantID", "questionID", "answer_content_oriLAN", "answer_content_EN"]
CONTENT_SIM_THRESHOLD = 0.95   # ratio scale 0-1, fuzz.ratio returns 0-100

In [2]:
import sqlite3
import re
import pandas as pd
from rapidfuzz import fuzz

def load_etl(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        """SELECT answerID, notegroupID, questionID, participantID,
                  answer_content_oriLAN, answer_content_EN
           FROM answers
           WHERE notegroupID BETWEEN ? AND ?""",
        con, params=(id_min, id_max)
    )
    con.close()
    df["answerID"] = df["answerID"].astype(int)
    return df.set_index("answerID")

def load_benchmark(xlsx_path, sheet, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, dtype=str)
    df["notegroupID"] = df["notegroupID"].astype(int)
    df["answerID"]    = df["answerID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    return df.set_index("answerID")

def load_id_map(csv_path, etl_col, bm_col):
    """Load a matched-pairs CSV and return etl->bm and bm->etl id dicts."""
    df = pd.read_csv(csv_path)
    df[etl_col] = df[etl_col].astype(int)
    df[bm_col]  = df[bm_col].astype(int)
    etl_to_bm = dict(zip(df[etl_col], df[bm_col]))
    bm_to_etl = dict(zip(df[bm_col], df[etl_col]))
    return etl_to_bm, bm_to_etl

def load_matched_answers(csv_path):
    df = pd.read_csv(csv_path)
    df["etl_answerID"] = df["etl_answerID"].astype(int)
    df["bm_answerID"]  = df["bm_answerID"].astype(int)
    return df

etl     = load_etl(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm      = load_benchmark(BENCHMARK_PATH, BENCHMARK_SHEET, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
matched = load_matched_answers(MATCHED_ANSWERS_CSV)

participant_etl_to_bm, _ = load_id_map(MATCHED_PARTICIPANTS_CSV, "etl_participantID", "bm_participantID")
question_etl_to_bm, _    = load_id_map(MATCHED_QUESTIONS_CSV,    "etl_questionID",    "bm_questionID")

matched_etl = set(matched["etl_answerID"])
matched_bm  = set(matched["bm_answerID"])
etl_only    = sorted(set(etl.index) - matched_etl)
bm_only     = sorted(set(bm.index)  - matched_bm)

print(f"ETL records   : {len(etl)}")
print(f"BM records    : {len(bm)}")
print(f"Matched pairs : {len(matched)}")
print(f"ETL-only (FP) : {len(etl_only)}")
print(f"BM-only  (FN) : {len(bm_only)}")

ETL records   : 1222
BM records    : 1221
Matched pairs : 1194
ETL-only (FP) : 28
BM-only  (FN) : 27


In [3]:
def normalise_str(val):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    s = str(val).strip().lower()
    s = re.sub(r'\s*\n\s*', '\n', s)
    return s

def normalise_id(val):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    try:
        return int(float(str(val).strip()))
    except (ValueError, TypeError):
        return None

def normalise_mapped_id(val, id_map):
    """Translate an ETL id to BM space using id_map, return as string or None."""
    qid = normalise_id(val)
    if qid is None:
        return None
    mapped = id_map.get(qid)
    return str(mapped) if mapped is not None else str(qid)

def compute_counts(etl_val, bm_val, sim_threshold=None):
    """
    Return (TP, FP, FN, TN) for one field comparison.
    If sim_threshold is set, treat as a match when fuzz.ratio/100 >= sim_threshold
    instead of requiring exact equality.
    """
    e = normalise_str(etl_val)
    b = normalise_str(bm_val)
    if e is not None and b is not None:
        if sim_threshold is not None:
            is_match = (fuzz.ratio(e, b) / 100) >= sim_threshold
        else:
            is_match = (e == b)
        return (1, 0, 0, 0) if is_match else (0, 1, 1, 0)
    if e is not None and b is None:
        return (0, 1, 0, 0)
    if e is None and b is not None:
        return (0, 0, 1, 0)
    return (0, 0, 0, 1)

def safe_div(num, den):
    return round(num / den, 4) if den > 0 else None

def metrics_from_counts(TP, FP, FN, TN):
    accuracy  = safe_div(TP + TN, TP + FP + FN + TN)
    precision = safe_div(TP, TP + FP)
    recall    = safe_div(TP, TP + FN)
    f1 = round(2 * precision * recall / (precision + recall), 4) \
         if precision and recall and (precision + recall) > 0 else None
    return dict(TP=TP, FP=FP, FN=FN, TN=TN,
                accuracy=accuracy, precision=precision, recall=recall, F1=f1)

In [4]:
totals = {f: dict(TP=0, FP=0, FN=0, TN=0) for f in EVAL_FIELDS}

for _, row in matched.iterrows():
    ei = row["etl_answerID"]
    bi = row["bm_answerID"]
    for field in EVAL_FIELDS:
        etl_val = etl.at[ei, field] if field in etl.columns else None
        bm_val  = bm.at[bi, field]  if field in bm.columns  else None

        if field == "participantID":
            etl_val = normalise_mapped_id(etl_val, participant_etl_to_bm)
            bm_val  = normalise_str(str(bm_val)) if not pd.isna(bm_val) and str(bm_val).strip() not in ("", "None", "nan") else None
            tp, fp, fn, tn = compute_counts(etl_val, bm_val)
        elif field == "questionID":
            etl_val = normalise_mapped_id(etl_val, question_etl_to_bm)
            bm_val  = normalise_str(str(bm_val)) if not pd.isna(bm_val) and str(bm_val).strip() not in ("", "None", "nan") else None
            tp, fp, fn, tn = compute_counts(etl_val, bm_val)
        elif field in ("answer_content_oriLAN", "answer_content_EN"):
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, sim_threshold=CONTENT_SIM_THRESHOLD)
        else:
            tp, fp, fn, tn = compute_counts(etl_val, bm_val)

        totals[field]["TP"] += tp; totals[field]["FP"] += fp
        totals[field]["FN"] += fn; totals[field]["TN"] += tn

# ETL-only rows → every field counts as FP
for ei in etl_only:
    for field in EVAL_FIELDS:
        totals[field]["FP"] += 1

# BM-only rows → every field counts as FN
for bi in bm_only:
    for field in EVAL_FIELDS:
        totals[field]["FN"] += 1

In [5]:
rows = []
for field in EVAL_FIELDS:
    m = metrics_from_counts(**totals[field])
    rows.append({"field": field, **m})

overall = {k: sum(totals[f][k] for f in EVAL_FIELDS) for k in ("TP","FP","FN","TN")}
m_all = metrics_from_counts(**overall)
rows.append({"field": "OVERALL", **m_all})

results_df = pd.DataFrame(rows).set_index("field")
results_df

,TP,FP,FN,TN,accuracy,precision,recall,F1
field,,,,,,,,
participantID,1149,34,31,39,0.9481,0.9713,0.9737,0.9725
questionID,1193,29,28,0,0.9544,0.9763,0.9771,0.9767
answer_content_oriLAN,1183,39,38,0,0.9389,0.9681,0.9689,0.9685
answer_content_EN,108,28,27,1086,0.9560,0.7941,0.8000,0.7970
OVERALL,3633,130,124,1125,0.9493,0.9655,0.9670,0.9662
